In [ ]:
from google.colab import files
uploaded = files.upload()



KeyboardInterrupt: 

In [ ]:
!pip install transformers torch pandas scikit-learn tqdm

import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizer
import torch, os

# Make sure output folder exists
os.makedirs("data", exist_ok=True)

# 1. Load Dataset
real = pd.read_csv("True.csv")
fake = pd.read_csv("Fake.csv")

# (Optional) Use smaller subset for faster testing
real = real.sample(3000, random_state=42)
fake = fake.sample(3000, random_state=42)

# Add label column
real["label"] = 1   # REAL = 1
fake["label"] = 0   # FAKE = 0

# Combine datasets
data = pd.concat([real, fake], axis=0).reset_index(drop=True)

# 2. Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\[.*?\]", "", text)
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>+", "", text)
    text = re.sub(r"[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\w*\d\w*", "", text)
    return text

data["text"] = data["text"].apply(clean_text)

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    data["text"], data["label"], test_size=0.2, random_state=42
)

# Save CSVs
pd.DataFrame({"text": X_train, "label": y_train}).to_csv("data/train.csv", index=False)
pd.DataFrame({"text": X_test, "label": y_test}).to_csv("data/test.csv", index=False)

# 4. Tokenization with DistilBERT
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(
    list(X_train), truncation=True, padding=True, max_length=256, return_tensors="pt"
)
test_encodings = tokenizer(
    list(X_test), truncation=True, padding=True, max_length=256, return_tensors="pt"
)

# Convert labels to torch tensors
train_labels = torch.tensor(y_train.values)
test_labels = torch.tensor(y_test.values)

# Save
torch.save((train_encodings, train_labels), "data/train_tokenized.pt")
torch.save((test_encodings, test_labels), "data/test_tokenized.pt")

print("Preprocessing completed")
print("Train size:", len(train_labels))
print("Test size:", len(test_labels))


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertForSequenceClassification, get_scheduler
from torch.optim import AdamW
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from transformers.tokenization_utils_base import BatchEncoding

# 1. Load Preprocessed Data
with torch.serialization.safe_globals([BatchEncoding]):
    train_encodings, train_labels = torch.load("data/train_tokenized.pt", weights_only=False)
    test_encodings, test_labels = torch.load("data/test_tokenized.pt", weights_only=False)

# 2. Dataset Class
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# 3. DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# 4. Load DistilBERT Model
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.to(device)

# 5. Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# 6. Training Loop
epochs = 3
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    print(f" Epoch {epoch+1} completed with loss: {loss.item()}")

# 7. Evaluation
model.eval()
preds, labels = [], []
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        preds.extend(predictions.cpu().numpy())
        labels.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(labels, preds)
print("🎯 Test Accuracy:", acc)
print("\nClassification Report:\n", classification_report(labels, preds, target_names=["Fake", "Real"]))

# 8. Save Model
model.save_pretrained("saved_model_distilbert")
print("Model training completed and saved to 'saved_model_distilbert/'")


Using device: cuda


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
 33%|███▎      | 1202/3600 [01:59<04:08,  9.66it/s]

 Epoch 1 completed with loss: 0.0012565285433083773


 67%|██████▋   | 2402/3600 [04:01<02:00,  9.97it/s]

 Epoch 2 completed with loss: 0.0032720095477998257


100%|██████████| 3600/3600 [06:02<00:00,  9.88it/s]

 Epoch 3 completed with loss: 0.00025477478629909456
🎯 Test Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

        Fake       1.00      1.00      1.00       587
        Real       1.00      1.00      1.00       613

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200

Model training completed and saved to 'saved_model_distilbert/'


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
pip install tldextract


In [ ]:
import pandas as pd
import tldextract
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch


# 1. Load Domain List (Whitelist + Blacklist)

domain_df = pd.read_csv("domain_list.csv")
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

def check_domain(url):
    """Check if URL is in whitelist, blacklist, or unknown"""
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")


# 2. Load Trained Fake News Model

model = DistilBertForSequenceClassification.from_pretrained("saved_model_distilbert")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    """Run ML model (BERT) for Fake/Real classification"""
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if prediction == 1 else "fake"

# 3. Simple Metadata Checks
def check_metadata(article):
    """
    Perform lightweight metadata checks
    (Can expand later with publication date, cross-reference APIs, etc.)
    """
    suspicious = []

    # Rule 1: Very short title/text
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")

    # Rule 2: Excessive use of ALL CAPS
    if article.isupper():
        suspicious.append("All Caps")

    # Rule 3: Missing author/publisher (if you pass metadata in future)
    # Here just a placeholder
    # if not article_author:
    #     suspicious.append("Missing author")

    return "suspect" if suspicious else "clean"

# 4. Final Verification Function

def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)

    # Combine results
    if domain_status == "blacklist":
        return " Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        if ml_prediction == "fake":
            return " Suspicious (Trusted source but ML flagged as fake)"
        else:
            return " Real News (Trusted Source)"
    else:  # Unknown domain
        if ml_prediction == "fake" or meta_status == "suspect":
            return " Fake News (Unknown Domain + Flags)"
        else:
            return " Likely Real (Unknown Domain + Clean Metadata)"
# 5. Example Usage
url1 = "https://www.bbc.com/news/world-asia-india-12345"
text1 = "India successfully launched its new satellite today."
print(verify_news(url1, text1))

url2 = "http://fakenews.com/story/999"
text2 = "ALIENS HAVE LANDED IN DELHI AND ARE TAKING OVER PARLIAMENT"
print(verify_news(url2, text2))

url3 = "https://randomblog.net/article"
text3 = "Breaking: Free gold for everyone in Hyderabad market."
print(verify_news(url3, text3))


In [ ]:
import pandas as pd
import tldextract
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch


# 1. Load Domain List (Whitelist + Blacklist)

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

# Check columns
if not {"domain", "category"}.issubset(domain_df.columns):
    raise ValueError("CSV must have 'domain' and 'category' columns")

# Create lookup dictionary
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

def check_domain(url):
    """Check if URL is in whitelist, blacklist, or unknown"""
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")


# 2. Load Trained Fake News Model

model = DistilBertForSequenceClassification.from_pretrained("saved_model_distilbert")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    """Run ML model (BERT) for Fake/Real classification"""
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if prediction == 1 else "fake"

# 3. Simple Metadata Checks
def check_metadata(article):
    """Lightweight metadata checks (can expand later)"""
    suspicious = []

    # Rule 1: Very short text
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")

    # Rule 2: All uppercase
    if article.isupper():
        suspicious.append("All Caps")

    return "suspect" if suspicious else "clean"


# 4. Final Verification Function
def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)

    # Combine results
    if domain_status == "blacklist":
        return " Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        if ml_prediction == "fake":
            return " Suspicious (Trusted source but ML flagged as fake)"
        else:
            return " Real News (Trusted Source)"
    else:  # Unknown domain
        if ml_prediction == "fake" or meta_status == "suspect":
            return "Fake News (Unknown Domain + Flags)"
        else:
            return " Likely Real (Unknown Domain + Clean Metadata)"


# 5. Example Usage
url1 = "https://www.bbc.com/news/world-asia-india-12345"
text1 = "India successfully launched its new satellite today."
print(verify_news(url1, text1))

url2 = "http://fakenews.com/story/999"
text2 = "ALIENS HAVE LANDED IN DELHI AND ARE TAKING OVER PARLIAMENT"
print(verify_news(url2, text2))

url3 = "https://randomblog.net/article"
text3 = "Breaking: Free gold for everyone in Hyderabad market."
print(verify_news(url3, text3))


In [ ]:
import pandas as pd

# Read CSV safely, skip bad lines
domain_df = pd.read_csv("domain_list.csv", encoding="utf-8", on_bad_lines='skip')

# Strip spaces from columns
domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()

# Only keep rows where category is whitelist or blacklist
domain_df = domain_df[domain_df['category'].isin(['whitelist', 'blacklist'])]

print(domain_df.head(10))


In [ ]:
import os

# Check if the file exists
if os.path.exists("domain_list.csv"):
    os.remove("domain_list.csv")
    print("domain_list.csv has been deleted.")
else:
    print("domain_list.csv does not exist.")


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving domain_list.csv to domain_list.csv


In [ ]:

# 0. Install required packages (run in Colab)

!pip install -q tldextract transformers torch


# 1. Imports

import pandas as pd
import tldextract
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import os

# 2. Load Domain List (Whitelist + Blacklist) safely

if not os.path.exists("domain_list.csv"):
    raise FileNotFoundError("domain_list.csv not found in current directory.")

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

# Strip whitespace from columns
domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()

# Validate columns
if not {"domain", "category"}.issubset(domain_df.columns):
    raise ValueError("CSV must have 'domain' and 'category' columns")

# Create lookup dictionary
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))


# 3. Domain Check Pipeline

def check_domain(url):
    """Check if URL is in whitelist, blacklist, or unknown"""
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")


# 4. Load Trained Fake News Model

model_path = "saved_model_distilbert"
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model folder '{model_path}' not found.")

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    """Run ML model (BERT) for Fake/Real classification"""
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=256
    )
    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if prediction == 1 else "fake"

# 5. Metadata Check Pipeline

def check_metadata(article):
    """Lightweight metadata checks"""
    suspicious = []

    if len(article.strip().split()) < 5:
        suspicious.append("Too short")

    if article.isupper():
        suspicious.append("All Caps")

    return "suspect" if suspicious else "clean"


# 6. Final Verification Function

def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)

    # Combine results
    if domain_status == "blacklist":
        return " Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        if ml_prediction == "fake":
            return " Suspicious (Trusted source but ML flagged as fake)"
        else:
            return " Real News (Trusted Source)"
    else:  # Unknown domain
        if ml_prediction == "fake" or meta_status == "suspect":
            return " Fake News (Unknown Domain + Flags)"
        else:
            return " Likely Real (Unknown Domain + Clean Metadata)"


# 7. Example Usage

examples = [
    ("https://www.bbc.com/news/world-asia-india-12345",
     "India successfully launched its new satellite today."),

    ("http://fakenews.com/story/999",
     "ALIENS HAVE LANDED IN DELHI AND ARE TAKING OVER PARLIAMENT"),

    ("https://randomblog.net/article",
     "Breaking: Free gold for everyone in Hyderabad market.")
]

for url, text in examples:
    print(f"URL: {url}\nResult: {verify_news(url, text)}\n")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 4.2 MB/s eta 0:00:00


/tmp/ipython-input-836395805.py:40: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = tldextract.extract(url).registered_domain


URL: https://www.bbc.com/news/world-asia-india-12345
Result:  Suspicious (Trusted source but ML flagged as fake)

URL: http://fakenews.com/story/999
Result:  Fake News (Blacklisted Domain)

URL: https://randomblog.net/article
Result:  Fake News (Unknown Domain + Flags)



In [ ]:
# ==============================
# Next Step: Run full verification and save results
# ==============================

import pandas as pd

# Example list of URLs and their corresponding article texts
examples = [
    ("https://www.bbc.com/news/world-asia-india-12345",
     "India successfully launched its new satellite today."),
    ("http://fakenews.com/story/999",
     "ALIENS HAVE LANDED IN DELHI AND ARE TAKING OVER PARLIAMENT"),
    ("https://randomblog.net/article",
     "Breaking: Free gold for everyone in Hyderabad market.")
]

urls, texts = zip(*examples)

# Run pipelines
ml_preds = [run_ml_model(text) for text in texts]  # ML predictions
meta_flags = [check_metadata(text) for text in texts]  # Metadata check
domain_flags = [check_domain(url) for url in urls]  # Domain check

# Combine results
results = []
for url, ml, meta, domain in zip(urls, ml_preds, meta_flags, domain_flags):
    if domain == "blacklist":
        final = "❌ Fake News (Blacklisted Domain)"
    elif domain == "whitelist":
        final = "⚠️ Suspicious (Trusted source but ML flagged as fake)" if ml=="fake" else "✅ Real News (Trusted Source)"
    else:
        final = "❌ Fake News (Unknown Domain + Flags)" if ml=="fake" or meta=="suspect" else "✅ Likely Real (Unknown Domain + Clean Metadata)"
    results.append((url, final))

# Convert to DataFrame
df_results = pd.DataFrame(results, columns=["URL", "Verdict"])

# Save results to CSV
df_results.to_csv("news_verification_results.csv", index=False)
print("✅ Results saved to news_verification_results.csv")

# Display results
print(df_results)


✅ Results saved to news_verification_results.csv
                                               URL  \
0  https://www.bbc.com/news/world-asia-india-12345   
1                    http://fakenews.com/story/999   
2                   https://randomblog.net/article   

                                             Verdict  
0  ⚠️ Suspicious (Trusted source but ML flagged a...  
1                   ❌ Fake News (Blacklisted Domain)  
2               ❌ Fake News (Unknown Domain + Flags)  


/tmp/ipython-input-836395805.py:40: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = tldextract.extract(url).registered_domain


In [ ]:
!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 96.2 MB/s eta 0:00:00


In [ ]:
import streamlit as st
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import pandas as pd
import tldextract
import os

# ---------------------------
# Page Configuration
# ---------------------------
st.set_page_config(
    page_title="Fake News Verifier",
    page_icon="📰",
    layout="centered",
    initial_sidebar_state="expanded",
    menu_items={
        'About': "This app verifies news using domain, metadata, and BERT ML model."
    }
)

# Enable dark mode via Streamlit theme in config.toml or allow system theme
# Users can enable dark mode in Streamlit settings (top-right menu)

# ---------------------------
# Load Domain List
# ---------------------------
if not os.path.exists("domain_list.csv"):
    st.error("domain_list.csv not found in current directory.")
    st.stop()

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

# ---------------------------
# Domain Check Function
# ---------------------------
def check_domain(url):
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")

# ---------------------------
# Load ML Model
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "saved_model_distilbert"
if not os.path.exists(model_path):
    st.error(f"Model folder '{model_path}' not found.")
    st.stop()

model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if pred==1 else "fake"

# ---------------------------
# Metadata Check
# ---------------------------
def check_metadata(article):
    suspicious = []
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")
    if article.isupper():
        suspicious.append("All Caps")
    return "suspect" if suspicious else "clean"

# ---------------------------
# Verify News
# ---------------------------
def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)

    if domain_status == "blacklist":
        return "❌ Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        return "⚠️ Suspicious (Trusted source but ML flagged as fake)" if ml_prediction=="fake" else "✅ Real News (Trusted Source)"
    else:
        return "❌ Fake News (Unknown Domain + Flags)" if ml_prediction=="fake" or meta_status=="suspect" else "✅ Likely Real (Unknown Domain + Clean Metadata)"

# ---------------------------
# Streamlit UI
# ---------------------------
st.title("📰 Fake News Verifier")
st.markdown("Enter a URL and the article text below to verify whether the news is real or fake.")

with st.form("verify_form"):
    url_input = st.text_input("Enter URL")
    article_input = st.text_area("Enter Article Text", height=150)
    submitted = st.form_submit_button("Verify News")

if submitted:
    if url_input.strip() == "" or article_input.strip() == "":
        st.warning("Please enter both URL and article text.")
    else:
        with st.spinner("Analyzing..."):
            result = verify_news(url_input, article_input)
        st.success(f"Result: {result}")


2025-09-05 05:42:15.492 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-05 05:42:16.945 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-05 05:42:17.040 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-09-05 05:42:17.041 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-05 05:42:17.042 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-05 05:42:17.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-05 05:42:17.043 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
# 0. Install required packages
!pip install -q streamlit pyngrok transformers torch tldextract pandas

# 1. Save Streamlit app to a file
%%writefile app.py
import streamlit as st
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import pandas as pd
import tldextract
import os

# ---------------------------
# Page Configuration
# ---------------------------
st.set_page_config(
    page_title="Fake News Verifier",
    page_icon="📰",
    layout="centered",
    initial_sidebar_state="expanded",
    menu_items={
        'About': "This app verifies news using domain, metadata, and BERT ML model."
    }
)

st.title("📰 Fake News Verifier")
st.markdown("Enter a URL and the article text below to verify whether the news is real or fake.")

# ---------------------------
# Load Domain List
# ---------------------------
if not os.path.exists("domain_list.csv"):
    st.error("domain_list.csv not found in current directory.")
    st.stop()

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

def check_domain(url):
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")

# ---------------------------
# Load ML Model
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "saved_model_distilbert"
if not os.path.exists(model_path):
    st.error(f"Model folder '{model_path}' not found.")
    st.stop()

model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if pred==1 else "fake"

def check_metadata(article):
    suspicious = []
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")
    if article.isupper():
        suspicious.append("All Caps")
    return "suspect" if suspicious else "clean"

def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)
    if domain_status == "blacklist":
        return "❌ Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        return "⚠️ Suspicious (Trusted source but ML flagged as fake)" if ml_prediction=="fake" else "✅ Real News (Trusted Source)"
    else:
        return "❌ Fake News (Unknown Domain + Flags)" if ml_prediction=="fake" or meta_status=="suspect" else "✅ Likely Real (Unknown Domain + Clean Metadata)"

# ---------------------------
# Streamlit Form
# ---------------------------
with st.form("verify_form"):
    url_input = st.text_input("Enter URL")
    article_input = st.text_area("Enter Article Text", height=150)
    submitted = st.form_submit_button("Verify News")

if submitted:
    if url_input.strip() == "" or article_input.strip() == "":
        st.warning("Please enter both URL and article text.")
    else:
        with st.spinner("Analyzing..."):
            result = verify_news(url_input, article_input)
        st.success(f"Result: {result}")


UsageError: Line magic function `%%writefile` not found.


In [ ]:
# 2. Launch Streamlit app with pyngrok
from pyngrok import ngrok

# Kill any previous tunnels
ngrok.kill()

# Run Streamlit in the background
get_ipython().system_raw("streamlit run app.py &")

# Open public URL
public_url = ngrok.connect(8501)
print(f"Click this link to open your Streamlit app: {public_url}")


In [ ]:
# ===============================
# Single-step Streamlit Launcher
# ===============================

# 1. Install required packages
!pip install -q streamlit pyngrok transformers torch tldextract pandas

# 2. Save Streamlit app to a file
%%writefile app.py
import streamlit as st
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import pandas as pd
import tldextract
import os

# Page config
st.set_page_config(
    page_title="Fake News Verifier",
    page_icon="📰",
    layout="centered"
)

st.title("📰 Fake News Verifier")
st.markdown("Enter a URL and the article text below to verify whether the news is real or fake.")

# Load domain list
if not os.path.exists("domain_list.csv"):
    st.error("domain_list.csv not found in current directory.")
    st.stop()

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

def check_domain(url):
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")

# Load ML model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "saved_model_distilbert"
if not os.path.exists(model_path):
    st.error(f"Model folder '{model_path}' not found.")
    st.stop()

model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if pred==1 else "fake"

def check_metadata(article):
    suspicious = []
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")
    if article.isupper():
        suspicious.append("All Caps")
    return "suspect" if suspicious else "clean"

def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)
    if domain_status == "blacklist":
        return "❌ Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        return "⚠️ Suspicious (Trusted source but ML flagged as fake)" if ml_prediction=="fake" else "✅ Real News (Trusted Source)"
    else:
        return "❌ Fake News (Unknown Domain + Flags)" if ml_prediction=="fake" or meta_status=="suspect" else "✅ Likely Real (Unknown Domain + Clean Metadata)"

# Streamlit form
with st.form("verify_form"):
    url_input = st.text_input("Enter URL")
    article_input = st.text_area("Enter Article Text", height=150)
    submitted = st.form_submit_button("Verify News")

if submitted:
    if url_input.strip() == "" or article_input.strip() == "":
        st.warning("Please enter both URL and article text.")
    else:
        with st.spinner("Analyzing..."):
            result = verify_news(url_input, article_input)
        st.success(f"Result: {result}")

# ------------------------------
# 3. Launch Streamlit with pyngrok
# ------------------------------
from pyngrok import ngrok
import time
ngrok.kill()
get_ipython().system_raw("streamlit run app.py &")
time.sleep(5)  # wait a few seconds for server to start
public_url = ngrok.connect(8501)
print(f"✅ Click this link to open your Streamlit app: {public_url}")


UsageError: Line magic function `%%writefile` not found.


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import torch
import pandas as pd
import tldextract
import os

st.set_page_config(
    page_title="Fake News Verifier",
    page_icon="📰",
    layout="centered"
)

st.title("📰 Fake News Verifier")
st.markdown("Enter a URL and the article text below to verify whether the news is real or fake.")

# Load domain list
if not os.path.exists("domain_list.csv"):
    st.error("domain_list.csv not found in current directory.")
    st.stop()

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

def check_domain(url):
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")

# Load ML model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = "saved_model_distilbert"
if not os.path.exists(model_path):
    st.error(f"Model folder '{model_path}' not found.")
    st.stop()

model = DistilBertForSequenceClassification.from_pretrained(model_path).to(device)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model.eval()

def run_ml_model(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=256).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if pred==1 else "fake"

def check_metadata(article):
    suspicious = []
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")
    if article.isupper():
        suspicious.append("All Caps")
    return "suspect" if suspicious else "clean"

def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)
    if domain_status == "blacklist":
        return "❌ Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        return "⚠️ Suspicious (Trusted source but ML flagged as fake)" if ml_prediction=="fake" else "✅ Real News (Trusted Source)"
    else:
        return "❌ Fake News (Unknown Domain + Flags)" if ml_prediction=="fake" or meta_status=="suspect" else "✅ Likely Real (Unknown Domain + Clean Metadata)"

# Streamlit form
with st.form("verify_form"):
    url_input = st.text_input("Enter URL")
    article_input = st.text_area("Enter Article Text", height=150)
    submitted = st.form_submit_button("Verify News")

if submitted:
    if url_input.strip() == "" or article_input.strip() == "":
        st.warning("Please enter both URL and article text.")
    else:
        with st.spinner("Analyzing..."):
            result = verify_news(url_input, article_input)
        st.success(f"Result: {result}")


Writing app.py


In [ ]:
!pip install -q pyngrok streamlit

from pyngrok import ngrok
import time

# Kill previous tunnels if any
ngrok.kill()

# Run Streamlit in the background
get_ipython().system_raw("streamlit run app.py &")

# Wait a few seconds for Streamlit server to start
time.sleep(5)

# Create a public URL
public_url = ngrok.connect(8501)
print(f"✅ Click this link to open your Streamlit app: {public_url}")


ERROR:pyngrok.process.ngrok:t=2025-09-05T05:49:54+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-09-05T05:49:54+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-09-05T05:49:54+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
files.download("app.py")
files.download("saved_model_distilbert/config.json")  # example for model file


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -r saved_model_distilbert.zip saved_model_distilbert
files.download("saved_model_distilbert.zip")


  adding: saved_model_distilbert/ (stored 0%)
  adding: saved_model_distilbert/model.safetensors (deflated 8%)
  adding: saved_model_distilbert/config.json (deflated 45%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ====================================
# 1. Install Required Packages
# ====================================
!pip install -q transformers datasets torch scikit-learn pandas

# ====================================
# 2. Imports
# ====================================
import pandas as pd
import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# ====================================
# 3. Load Your New Dataset
# ====================================
# Make sure new_data.csv is uploaded in Colab
df = pd.read_csv("new_data.csv")  # columns: 'text', 'label' (0=fake, 1=real)
dataset = Dataset.from_pandas(df)

# Optional: split into train/test
dataset = dataset.train_test_split(test_size=0.1)
dataset = DatasetDict({"train": dataset['train'], "test": dataset['test']})

# ====================================
# 4. Tokenization with max_length=512
# ====================================
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=512)  # max_length increased

dataset = dataset.map(tokenize, batched=True)
dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# ====================================
# 5. Load Pre-trained Model
# ====================================
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

# ====================================
# 6. Training Setup
# ====================================
training_args = TrainingArguments(
    output_dir="saved_model_distilbert_new",
    num_train_epochs=3,
    per_device_train_batch_size=8,   # Reduce batch size if memory issues occur
    per_device_eval_batch_size=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=50,
    learning_rate=5e-5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=2
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ====================================
# 7. Train the Model
# ====================================
trainer.train()

# ====================================
# 8. Save the Fine-Tuned Model
# ====================================
trainer.save_model("saved_model_distilbert_new")
print("✅ Model saved as 'saved_model_distilbert_new'")

# ====================================
# 9. Test with Example Articles
# ====================================
def run_ml_model(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        pred = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if pred==1 else "fake"

examples = [
    "Breaking news: India launches new satellite successfully.",
    "ALIENS HAVE LANDED AND TAKEN OVER THE CITY HALL"
]

for text in examples:
    print(f"Text: {text}\nPrediction: {run_ml_model(text)}\n")


FileNotFoundError: [Errno 2] No such file or directory: 'new_data.csv'

In [ ]:
# ====================================
# 0. Install Required Packages (run in Colab or local)
# ====================================
!pip install -q transformers torch scikit-learn tqdm

# ====================================
# 1. Imports
# ====================================
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, get_scheduler
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

# ====================================
# 2. Load Preprocessed Data
# ====================================
# Make sure these files exist and were saved properly as (encodings_dict, labels_tensor)
train_encodings, train_labels = torch.load("data/train_tokenized.pt")
test_encodings, test_labels = torch.load("data/test_tokenized.pt")

# ====================================
# 3. Dataset Class
# ====================================
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# ====================================
# 4. DataLoaders
# ====================================
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, pin_memory=True)

# ====================================
# 5. Load DistilBERT Model
# ====================================
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to(device)

# ====================================
# 6. Optimizer & Scheduler
# ====================================
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# ====================================
# 7. Training Loop
# ====================================
epochs = 3
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    print(f"Epoch {epoch+1} completed with loss: {loss.item()}")

# ====================================
# 8. Evaluation
# ====================================
model.eval()
preds, labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        preds.extend(predictions.cpu().numpy())
        labels_list.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(labels_list, preds)
print("🎯 Test Accuracy:", acc)
print("\nClassification Report:\n", classification_report(labels_list, preds, target_names=["Fake", "Real"]))

# ====================================
# 9. Save Model & Tokenizer
# ====================================
model.save_pretrained("saved_model_distilbert")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer.save_pretrained("saved_model_distilbert")
print("✅ Model and tokenizer saved to 'saved_model_distilbert/'")


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL transformers.tokenization_utils_base.BatchEncoding was not an allowed global by default. Please use `torch.serialization.add_safe_globals([transformers.tokenization_utils_base.BatchEncoding])` or the `torch.serialization.safe_globals([transformers.tokenization_utils_base.BatchEncoding])` context manager to allowlist this global if you trust this c

In [ ]:
# ====================================
# 0. Install Required Packages
# ====================================
!pip install -q transformers torch scikit-learn tqdm

# ====================================
# 1. Imports
# ====================================
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, get_scheduler
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

# ====================================
# 2. Load Raw Dataset (CSV)
# ====================================
import pandas as pd

df = pd.read_csv("new_data.csv")  # columns: 'text', 'label' (0=fake, 1=real)

# ====================================
# 3. Tokenization and Conversion to Tensors
# ====================================
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
max_len = 512  # Increased max token length

# Tokenize and convert to tensors
encodings = tokenizer(list(df["text"]), padding=True, truncation=True, max_length=max_len, return_tensors="pt")
labels = torch.tensor(df["label"].values)

# Optional: train/test split
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(range(len(labels)), test_size=0.1, random_state=42)

train_encodings = {k: v[train_idx] for k, v in encodings.items()}
train_labels = labels[train_idx]

test_encodings = {k: v[test_idx] for k, v in encodings.items()}
test_labels = labels[test_idx]

# ====================================
# 4. Dataset Class
# ====================================
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# ====================================
# 5. DataLoaders
# ====================================
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, pin_memory=True)

# ====================================
# 6. Load DistilBERT Model
# ====================================
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to(device)

# ====================================
# 7. Optimizer & Scheduler
# ====================================
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# ====================================
# 8. Training Loop
# ====================================
epochs = 3
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    print(f"Epoch {epoch+1} completed with loss: {loss.item()}")

# ====================================
# 9. Evaluation
# ====================================
model.eval()
preds, labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        preds.extend(predictions.cpu().numpy())
        labels_list.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(labels_list, preds)
print("🎯 Test Accuracy:", acc)
print("\nClassification Report:\n", classification_report(labels_list, preds, target_names=["Fake", "Real"]))

# ====================================
# 10. Save Model & Tokenizer
# ====================================
model.save_pretrained("saved_model_distilbert")
tokenizer.save_pretrained("saved_model_distilbert")
print("✅ Model and tokenizer saved to 'saved_model_distilbert/'")


FileNotFoundError: [Errno 2] No such file or directory: 'new_data.csv'

In [ ]:
# ====================================
# 0. Install Required Packages
# ====================================
!pip install -q transformers torch scikit-learn tqdm

# ====================================
# 1. Imports
# ====================================
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, get_scheduler
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

# ====================================
# 2. Load Tokenized Data Safely
# ====================================
from transformers import BatchEncoding

# Use safe_globals to allow BatchEncoding unpickling
with torch.serialization.safe_globals([BatchEncoding]):
    train_encodings, train_labels = torch.load("train_tokenized.pt", weights_only=False)
    test_encodings, test_labels = torch.load("test_tokenized.pt", weights_only=False)

# Ensure labels are tensors
if not isinstance(train_labels, torch.Tensor):
    train_labels = torch.tensor(train_labels)
if not isinstance(test_labels, torch.Tensor):
    test_labels = torch.tensor(test_labels)

# ====================================
# 3. Dataset Class
# ====================================
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# ====================================
# 4. DataLoaders
# ====================================
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, pin_memory=True)

# ====================================
# 5. Load DistilBERT Model
# ====================================
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to(device)

# ====================================
# 6. Optimizer & Scheduler
# ====================================
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# ====================================
# 7. Training Loop
# ====================================
epochs = 3
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    print(f"Epoch {epoch+1} completed with loss: {loss.item()}")

# ====================================
# 8. Evaluation
# ====================================
model.eval()
preds, labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        preds.extend(predictions.cpu().numpy())
        labels_list.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(labels_list, preds)
print("🎯 Test Accuracy:", acc)
print("\nClassification Report:\n", classification_report(labels_list, preds, target_names=["Fake", "Real"]))

# ====================================
# 9. Save Model & Tokenizer
# ====================================
model.save_pretrained("saved_model_distilbert")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer.save_pretrained("saved_model_distilbert")
print("✅ Model and tokenizer saved to 'saved_model_distilbert/'")


FileNotFoundError: [Errno 2] No such file or directory: 'train_tokenized.pt'

In [ ]:
# ====================================
# 0. Install Required Packages
# ====================================
!pip install -q transformers torch scikit-learn tqdm

# ====================================
# 1. Imports
# ====================================
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, get_scheduler
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

# ====================================
# 2. Load Tokenized Data Safely
# ====================================
from transformers import BatchEncoding

# Use safe_globals to allow BatchEncoding unpickling
with torch.serialization.safe_globals([BatchEncoding]):
    train_encodings, train_labels = torch.load("data/train_tokenized.pt", weights_only=False)
    test_encodings, test_labels = torch.load("data/test_tokenized.pt", weights_only=False)

# Ensure labels are tensors
if not isinstance(train_labels, torch.Tensor):
    train_labels = torch.tensor(train_labels)
if not isinstance(test_labels, torch.Tensor):
    test_labels = torch.tensor(test_labels)

# ====================================
# 3. Dataset Class
# ====================================
class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = NewsDataset(train_encodings, train_labels)
test_dataset = NewsDataset(test_encodings, test_labels)

# ====================================
# 4. DataLoaders
# ====================================
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, pin_memory=True)

# ====================================
# 5. Load DistilBERT Model
# ====================================
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print("Using device:", device)

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model.to(device)

# ====================================
# 6. Optimizer & Scheduler
# ====================================
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 3  # 3 epochs
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

# ====================================
# 7. Training Loop
# ====================================
epochs = 3
progress_bar = tqdm(range(num_training_steps))

model.train()
for epoch in range(epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        progress_bar.update(1)

    print(f"Epoch {epoch+1} completed with loss: {loss.item()}")

# ====================================
# 8. Evaluation
# ====================================
model.eval()
preds, labels_list = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=-1)
        preds.extend(predictions.cpu().numpy())
        labels_list.extend(batch["labels"].cpu().numpy())

acc = accuracy_score(labels_list, preds)
print("🎯 Test Accuracy:", acc)
print("\nClassification Report:\n", classification_report(labels_list, preds, target_names=["Fake", "Real"]))

# ====================================
# 9. Save Model & Tokenizer
# ====================================
model.save_pretrained("saved_model_distilbert")
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer.save_pretrained("saved_model_distilbert")
print("✅ Model and tokenizer saved to 'saved_model_distilbert/'")


Using device: cuda


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

100%|██████████| 3600/3600 [1:35:11<00:00,  1.59s/it]

 33%|███▎      | 1202/3600 [02:04<04:03,  9.84it/s]

Epoch 1 completed with loss: 7.795971760060638e-05



 67%|██████▋   | 2402/3600 [04:09<02:04,  9.64it/s]

Epoch 2 completed with loss: 5.0572642066981643e-05



100%|██████████| 3600/3600 [06:14<00:00,  8.14it/s]

Epoch 3 completed with loss: 1.9162558601237833e-05
🎯 Test Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

        Fake       1.00      1.00      1.00       587
        Real       1.00      1.00      1.00       613

    accuracy                           1.00      1200
   macro avg       1.00      1.00      1.00      1200
weighted avg       1.00      1.00      1.00      1200

✅ Model and tokenizer saved to 'saved_model_distilbert/'


In [ ]:
# 1. Zip the saved model folder
!zip -r saved_model_distilbert.zip saved_model_distilbert

# 2. Download the zip file
from google.colab import files
files.download("saved_model_distilbert.zip")


updating: saved_model_distilbert/ (stored 0%)
updating: saved_model_distilbert/model.safetensors (deflated 8%)
updating: saved_model_distilbert/config.json (deflated 45%)
  adding: saved_model_distilbert/vocab.txt (deflated 53%)
  adding: saved_model_distilbert/special_tokens_map.json (deflated 42%)
  adding: saved_model_distilbert/tokenizer_config.json (deflated 75%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ====================================
# 0. Install Required Packages (run in Colab only)
# ====================================
!pip install -q transformers tldextract torch pandas

# ====================================
# 1. Imports
# ====================================
import torch
import pandas as pd
import tldextract
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# ====================================
# 2. Load Domain List (Whitelist + Blacklist)
# ====================================
domain_file = "domain_list.csv"  # adjust path if needed
domain_df = pd.read_csv(domain_file)
domain_df['domain'] = domain_df['domain'].str.strip()
domain_df['category'] = domain_df['category'].str.strip()
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))

def check_domain(url):
    """Check if URL is in whitelist, blacklist, or unknown"""
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")

# ====================================
# 3. Load Trained Model & Tokenizer
# ====================================
model_path = "saved_model_distilbert"  # path to your downloaded model folder
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_path)
model.to(device)
model.eval()

def run_ml_model(text):
    """Run ML model (BERT) for Fake/Real classification"""
    inputs = tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=512
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        prediction = torch.argmax(outputs.logits, dim=-1).item()
    return "real" if prediction == 1 else "fake"

# ====================================
# 4. Metadata Checks
# ====================================
def check_metadata(article):
    """Lightweight metadata checks"""
    suspicious = []
    if len(article.strip().split()) < 5:
        suspicious.append("Too short")
    if article.isupper():
        suspicious.append("All Caps")
    return "suspect" if suspicious else "clean"

# ====================================
# 5. Final Verification Function
# ====================================
def verify_news(url, article_text):
    domain_status = check_domain(url)
    ml_prediction = run_ml_model(article_text)
    meta_status = check_metadata(article_text)

    if domain_status == "blacklist":
        return "❌ Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        if ml_prediction == "fake":
            return "⚠️ Suspicious (Trusted source but ML flagged as fake)"
        else:
            return "✅ Real News (Trusted Source)"
    else:  # Unknown domain
        if ml_prediction == "fake" or meta_status == "suspect":
            return "❌ Fake News (Unknown Domain + Flags)"
        else:
            return "✅ Likely Real (Unknown Domain + Clean Metadata)"

# ====================================
# 6. Example Usage
# ====================================
examples = [
    ("https://www.bbc.com/news/world-asia-india-12345",
     "India successfully launched its new satellite today."),
    ("http://fakenews.com/story/999",
     "ALIENS HAVE LANDED IN DELHI AND ARE TAKING OVER PARLIAMENT"),
    ("https://randomblog.net/article",
     "Breaking: Free gold for everyone in Hyderabad market.")
]

for url, text in examples:
    print(f"URL: {url}\nResult: {verify_news(url, text)}\n")


URL: https://www.bbc.com/news/world-asia-india-12345
Result: ⚠️ Suspicious (Trusted source but ML flagged as fake)

URL: http://fakenews.com/story/999
Result: ❌ Fake News (Blacklisted Domain)

URL: https://randomblog.net/article
Result: ❌ Fake News (Unknown Domain + Flags)



/tmp/ipython-input-348506733.py:25: DeprecationWarning: The 'registered_domain' property is deprecated and will be removed in the next major version. Use 'top_domain_under_public_suffix' instead, which has the same behavior but a more accurate name.
  domain = tldextract.extract(url).registered_domain


In [ ]:
import os
import torch
import torch.nn.functional as F
import pandas as pd
import tldextract
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# ==============================
# 1. Load Domain List
# ==============================
if not os.path.exists("domain_list.csv"):
    raise FileNotFoundError("domain_list.csv not found!")

try:
    domain_df = pd.read_csv("domain_list.csv", encoding="utf-8")
except UnicodeDecodeError:
    domain_df = pd.read_csv("domain_list.csv", encoding="latin1")

# Clean column spaces
domain_df["domain"] = domain_df["domain"].str.strip()
domain_df["category"] = domain_df["category"].str.strip()

if not {"domain", "category"}.issubset(domain_df.columns):
    raise ValueError("CSV must contain 'domain' and 'category' columns")

# Create lookup dictionary
domain_map = dict(zip(domain_df["domain"], domain_df["category"]))


def check_domain(url: str):
    """Check if domain is in whitelist/blacklist/unknown"""
    domain = tldextract.extract(url).registered_domain
    return domain_map.get(domain, "unknown")


# ==============================
# 2. Load ML Model (DistilBERT)
# ==============================
MODEL_PATH = "saved_model_distilbert"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model path '{MODEL_PATH}' not found!")

tokenizer = DistilBertTokenizer.from_pretrained(MODEL_PATH)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()


def run_ml_model(text: str, threshold: float = 0.7):
    """Run DistilBERT model and return softmax probabilities"""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512  # ✅ Increased to 512
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Softmax probabilities
    probs = F.softmax(outputs.logits, dim=-1).squeeze()
    confidence, predicted_class = torch.max(probs, dim=-1)

    label_map = {0: "Fake", 1: "Real"}
    prediction = label_map[predicted_class.item()]

    if confidence.item() < threshold:
        return "Suspicious (Low Confidence)", confidence.item(), probs.tolist()
    return prediction, confidence.item(), probs.tolist()


# ==============================
# 3. Metadata Check
# ==============================
def check_metadata(article: str):
    suspicious = []

    if len(article.strip().split()) < 5:
        suspicious.append("Too short")

    if article.isupper():
        suspicious.append("All Caps")

    return "suspect" if suspicious else "clean"


# ==============================
# 4. Final Verification
# ==============================
def verify_news(url: str, article_text: str):
    domain_status = check_domain(url)
    ml_prediction, ml_confidence, ml_probs = run_ml_model(article_text)
    meta_status = check_metadata(article_text)

    # Combine all 3 pipelines
    if domain_status == "blacklist":
        result = "❌ Fake News (Blacklisted Domain)"
    elif domain_status == "whitelist":
        if ml_prediction == "Fake":
            result = "⚠️ Suspicious (Trusted source but ML flagged as fake)"
        else:
            result = "✅ Real News (Trusted Source)"
    else:  # unknown domain
        if ml_prediction == "Fake" or meta_status == "suspect":
            result = "❌ Fake News (Unknown Domain + Flags)"
        elif ml_prediction.startswith("Suspicious"):
            result = "⚠️ Suspicious (Low Confidence)"
        else:
            result = "✅ Likely Real (Unknown Domain + Clean Metadata)"

    return {
        "result": result,
        "domain_check": domain_status,
        "ml_prediction": ml_prediction,
        "ml_confidence": ml_confidence,
        "ml_probabilities": {"Fake": ml_probs[0], "Real": ml_probs[1]},
        "metadata_check": meta_status,
    }


# ==============================
# 5. Example Usage
# ==============================
if __name__ == "__main__":
    tests = [
        ("https://www.bbc.com/news/world-asia-india-12345",
         "India successfully launched its new satellite today."),
        ("http://fakenews.com/story/999",
         "ALIENS HAVE LANDED IN DELHI AND ARE TAKING OVER PARLIAMENT"),
        ("https://randomblog.net/article",
         "Breaking: Free gold for everyone in Hyderabad market."),
        ("https://www.thehindu.com/news/national/",
         "Amaravati hackathon to promote culture of innovation and problem-solving")
    ]

    for url, text in tests:
        print(f"\n🔗 URL: {url}")
        print(verify_news(url, text))


In [ ]:
# ===============================
# 1. Install & Import Libraries
# ===============================
!pip install torch torchvision torchaudio transformers --quiet

import torch
import pandas as pd
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification

# ===============================
# 2. Load Trained Model
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
model.load_state_dict(torch.load("distilbert_fakenews.pth", map_location=device))
model.to(device)
model.eval()

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# ===============================
# 3. Load Whitelist & Blacklist
# ===============================
def load_domains(path):
    df = pd.read_csv(path)
    df.rename(columns={df.columns[0]: "source"}, inplace=True)
    domains = set(df["source"].astype(str).str.lower().str.strip())
    return domains

whitelist_domains = load_domains("whitelist_domains.csv")
blacklist_domains = load_domains("blacklist_domains.csv")

print(f"✅ Whitelist loaded: {len(whitelist_domains)} domains")
print(f"✅ Blacklist loaded: {len(blacklist_domains)} domains")

# ===============================
# 4. Hybrid Prediction Function
# ===============================
def hybrid_predict(text, source=None):
    """
    text   : News article content
    source : Domain name (e.g., 'thehindu.com')
    """
    # 1. Check domain lists
    if source:
        src = source.lower().strip()
        if src in whitelist_domains:
            return {"label": "REAL (trusted domain)", "confidence": 1.0}
        if src in blacklist_domains:
            return {"label": "FAKE (untrusted domain)", "confidence": 1.0}

    # 2. Otherwise, fall back to BERT model
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
        pred = probs.argmax()
        label = "REAL" if pred == 1 else "FAKE"
        confidence = float(probs[pred])

    return {"label": label, "confidence": confidence}

# ===============================
# 5. Example Usage
# ===============================
example_text = "The government has announced new policies for rural development."
example_source = "thehindu.com"  # try changing this to a blacklist domain

result = hybrid_predict(example_text, example_source)
print("Prediction:", result)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FileNotFoundError: [Errno 2] No such file or directory: 'distilbert_fakenews.pth'

In [ ]:
import pandas as pd
from transformers import BertTokenizer

# Load merged dataset
df = pd.read_csv("merged_dataset.csv")  # ISOT + other news dataset

# Basic text cleaning
df['text'] = df['text'].str.replace(r"http\S+|www\S+|https\S+", '', regex=True)  # remove URLs
df['text'] = df['text'].str.replace(r'<.*?>', '', regex=True)  # remove HTML tags
df['text'] = df['text'].str.lower().str.strip()  # lowercase and strip whitespace

# Optional: remove rows with empty text
df = df[df['text'].notnull() & (df['text'] != '')]

# Save cleaned dataset
df.to_csv("cleaned_news_dataset.csv", index=False)

print("✅ Cleaned dataset saved as cleaned_news_dataset.csv")


✅ Cleaned dataset saved as cleaned_news_dataset.csv
